## Config

In [ ]:
from dataclasses import dataclass, field
import numpy as np

SCALE_ZYX = np.array([1.625, 0.40625, 0.40625])

@dataclass
class config:

    gaussian_sigma: float = 1.0
    ds_factor_xy: int = 4
    thresh_rel: float = 0.32
    use_otsu: bool = True
    min_peak_dist: float = 2
    border_z: int = 0
    refine_rz: int = 2
    refine_ryx: int = 5
    nms_radius_um: float = 7.0
    scale: np.ndarray = field(default_factory=lambda: SCALE_ZYX.copy())
    max_nodes_per_frame: int = 0
    use_motion: bool = False
    gate: float = 6.0
    max_link_distance_um: float = 10.0
    prune_isolated: bool = True
    keep_strong_isolated_q: float = 0.0
    detect_divisions: bool = True
    div_min_count_gain: int = 1
    div_parent_dist_um: float = 9.0
    div_sister_dist_um: float = 9.0

In [ ]:
cfg = config()

## Detection

### Utils

In [ ]:
# Load volume .zarr, annotation .geff functions

import numpy as np
import pandas as pd
import zarr
from pathlib import Path

TRAIN_PATH = Path("/Users/ishan/Desktop/biohub_kaggle/data/train")
_ZC = {} # Cache 

def load_volume(vol_id: str):

    "Open the multiscale array '0', shape (T, Z, Y, X), cached by path."

    key = str(TRAIN_PATH / (vol_id + ".zarr"))
    if key not in _ZC:
        _ZC[key] = zarr.open(key, mode="r")["0"]
    return _ZC[key]


In [ ]:
from scipy.ndimage import gaussian_filter 
from typing import List
from skimage.filters import threshold_otsu
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree

def _mean_xy(
    volume: np.ndarray,
    f: int
):
    Z, Y, X = volume.shape; Y2, X2 = (Y//f)*f, (X//f)*f
    crop = volume[:, :Y2, :X2].astype(np.float32, copy=False)
    return crop.reshape(Z, Y2//f, f, X2//f, f).mean(axis=(2, 4))

def _threshold(
    smooth,
    cfg,
):
    
    bg = float(np.median(smooth)); hi = float(np.percentile(smooth, 99.9)) # BG - median, hi - 99.9% zones
    rel = bg + cfg.thresh_rel * max(hi - bg, 1e-6)

    if cfg.use_otsu:

        try: return max(float(threshold_otsu(smooth)), rel)
        except Exception: pass

    return max(float(np.percentile(smooth, 96.0)), rel)

def _peaks(
    smooth,
    threshold,
    d,
):
    
    return peak_local_max(
        smooth,
        min_distance=d,
        threshold_abs=threshold, 
        exclude_border=False
    ).astype(np.int32)

def _refine(
    vol, 
    zyx, 
    cfg
):

    """
    Refines detections by shifting the detected centers towards a center of mass 
    calculated from its local_reference_crop. 
    """

    Z, Y, X = vol.shape; z, y, x = (int(round(v)) for v in zyx)
    z0,z1 = max(0,z-cfg.refine_rz), min(Z,z+cfg.refine_rz+1)
    y0,y1 = max(0,y-cfg.refine_ryx), min(Y,y+cfg.refine_ryx+1)
    x0,x1 = max(0,x-cfg.refine_ryx), min(X,x+cfg.refine_ryx+1)
    crop = vol[z0:z1, y0:y1, x0:x1].astype(np.float32); bg = float(crop.min())
    w = np.clip(crop-bg, 0, None); s = float(w.sum())

    if s <= 0: return np.array([z,y,x], float), 0.0
    zz,yy,xx = np.mgrid[z0:z1, y0:y1, x0:x1]

    return np.array([(zz*w).sum(),(yy*w).sum(),(xx*w).sum()])/s, float(crop.max()-bg)

def _physical_nms(
    coords, 
    scores, 
    radius_um, 
    scale
):
    
    if len(coords) <= 1: return coords, scores

    pts = coords*scale[None,:]; order = np.argsort(-scores)
    tree = cKDTree(pts); killed = np.zeros(len(coords), bool); keep = []

    for i in order:
        if killed[i]: continue
        keep.append(int(i)); killed[tree.query_ball_point(pts[i], r=radius_um)] = True

    keep = np.array(keep); return coords[keep], scores[keep]

def detect(
    volume,
    cfg,
):
    
    f = cfg.ds_factor_xy
    pooled = _mean_xy(volume, f) if f > 1 else volume.astype(np.float32)
    sm = gaussian_filter(pooled, cfg.gaussian_sigma) if cfg.gaussian_sigma > 0.0 else pooled
    pk = _peaks(sm, _threshold(sm, cfg), cfg.min_peak_dist)

    if len(pk) == 0: return np.zeros((0,3)), np.zeros(0)

    full = pk.astype(float)
    full[:,1] = full[:,1]*f + (f-1)/2; full[:,2] = full[:,2]*f + (f-1)/2 # Expand x and y coordinates by f

    Z = volume.shape[0]; 
    coords, scores = [], []
    
    for p in full:
        if p[0] < cfg.border_z or p[0] > Z-1-cfg.border_z: continue
        c, s = _refine(volume, p, cfg); 
        coords.append(c); scores.append(s)
        
    if not coords: return np.zeros((0,3)), np.zeros(0)
        
    coords = np.array(coords); scores = np.array(scores)
    no_nms_coords, no_nms_scores = coords, scores

    coords, scores = _physical_nms(coords, scores, cfg.nms_radius_um, cfg.scale)
    
    if cfg.max_nodes_per_frame and len(coords) > cfg.max_nodes_per_frame:
        k = np.argsort(-scores)[:cfg.max_nodes_per_frame]; coords, scores = coords[k], scores[k]
        
    return coords, scores

### Linking

In [ ]:
from typing import List, Tuple
from scipy.optimize import linear_sum_assignment

def _link(
    prev_coords,
    curr_coords,
    cfg,
    prev_vel=None,
) -> List[Tuple]:
    
    if len(prev_coords) == 0 or len(curr_coords) == 0: return []

    P = prev_coords * cfg.scale[None, :]; C = curr_coords * cfg.scale[None, :]
    pred_traj = P + 0.5*(prev_vel if cfg.use_motion and prev_vel is not None else 0.0)
    N = len(P); M = len(C)

    def _hungarian_matching(
        prev_physical_coords_indices,
        curr_physical_coords_indices,
        gate
    ):
        
        raw_distances = np.sqrt(((P[prev_physical_coords_indices][:,None]-C[curr_physical_coords_indices][None])**2).sum(2))        
        pred_distances = np.sqrt(((pred_traj[prev_physical_coords_indices][:,None]-C[curr_physical_coords_indices][None])**2).sum(2))

        cost = np.where(raw_distances > gate, 1e9, pred_distances)
        ri, rc = linear_sum_assignment(cost) # ri, rc contain the pair of optimal indices after hungarian matching

        return [(int(prev_physical_coords_indices[r]), int(curr_physical_coords_indices[c])) for r,c in zip(ri,rc) if cost[r,c] < 1e9]

    gate = cfg.gate
    links = _hungarian_matching(np.arange(N), np.arange(M), gate)

    # Look for unmatched edges, retry with gate as max_link_distance_um
    used_p = {p for p, _ in links}; used_c = {c for _, c in links}
    fp = [i for i in range(N) if i not in used_p]; fc = [j for j in range(M) if j not in used_c]
    links += _hungarian_matching(fp, fc, cfg.max_link_distance_um)

    return links

def _divisions(
        prev_xyz, 
        curr_xyz, 
        links, 
        cfg
    ):

    if not cfg.detect_divisions or len(curr_xyz)-len(prev_xyz) < cfg.div_min_count_gain: return []
    
    P = prev_xyz*cfg.scale[None,:]; C = curr_xyz*cfg.scale[None,:]
    matched = {c for _,c in links}; parent_of = {c:p for p,c in links}
    free = [j for j in range(len(curr_xyz)) if j not in matched]
    
    if not free: return []
    ptree = cKDTree(P); extra = []
    
    for j in free:
        
        d, p = ptree.query(C[j], k=1)
        if d > cfg.div_parent_dist_um: continue
        sisters = [c for c,pp in parent_of.items() if pp == p]
        if not sisters: continue
        sis = min(sisters, key=lambda c: np.linalg.norm(C[c]-C[j]))
        if np.linalg.norm(C[sis]-C[j]) <= cfg.div_sister_dist_um: extra.append((int(p),int(j)))
    
    return extra

In [ ]:
import gc

COLS = ["dataset","row_type","node_id","t","z","y","x","source_id","target_id"]

def track_video(
    dataset: str,
    load_volume: callable,
    T: int,
    cfg: config,
):
    
    volume = load_volume(dataset)
    nid = 0
    nodes, edges, ndiv = [], [], 0
    prev_coords, prev_ids, prev_vel = [], [], None
    counts, node_score = [], {}
    
    for t in range(T):

        ## Fill the node details first
        vol_t = volume[t]

        coords, scores = detect(vol_t, cfg); del vol_t; gc.collect()
        ids = list(np.arange(nid, nid+len(coords))); nid += len(coords)

        for id, coord, s in zip(ids, coords, scores):
            nodes.append((dataset, "node", id, t, coord[0], coord[1], coord[2], -1, -1))
            node_score[id] = float(s) # add each node score, will be useful for pruning/keeping isolated edges
            
        ## Populate the edge information
        if t>0 and len(prev_ids):

            links = _link(prev_coords, coords, cfg, prev_vel)
            extras = _divisions(prev_coords, coords, links, cfg)
            vel = np.zeros((len(prev_coords), 3))

            # Add single links
            for p,c in links: # Get the links from previous to the current timestep
                edges.append((dataset, "edge", -1, -1, -1, -1, -1, prev_ids[p], ids[c]))
                vel[p] = (coords[c] - prev_coords[p])
            
            # Add bifurcations (sister cells)
            for p,c in extras:
                edges.append((dataset, "edge", -1, -1, -1, -1, -1, prev_ids[p], ids[c]))

            ndiv += len({p for p, _ in extras}) # Count the number of divisions in this timeframe transition
            nv = np.zeros((len(coords),3))

            for p, c in links: nv[c] = vel[p]
            prev_vel = nv

        else: 
            prev_vel = None

        prev_ids, prev_coords = ids, coords; counts.append(len(coords))
        
    nodes = pd.DataFrame(nodes, columns=COLS); edges = pd.DataFrame(edges, columns=COLS)

    if cfg.prune_isolated:

        used = set(edges.source_id) | set(edges.target_id)
        
        if cfg.keep_strong_isolated_q > 0 and node_score:
            fl = np.quantile(list(node_score.values()), cfg.keep_strong_isolated_q)
            used |= {i for i,s in node_score.items() if s >= fl} # Keep non connected nodes if the lie above keep_strong_isolated_q quantile
        
        nodes = nodes[nodes.node_id.isin(used)].reset_index(drop=True)
    
    return nodes, edges, dict(dataset=dataset, n_nodes=len(nodes), n_edges=len(edges),
                              cells_per_frame=float(np.mean(counts)) if counts else 0, n_div=ndiv, T=T)

In [ ]:
import time

TEST_PATH = TRAIN_PATH          # point at the test .zarr dir on Kaggle; locally we reuse train
OUT = "submission.csv"          # Kaggle runs with CWD=/kaggle/working, so this lands in the right place

def avail_T(zp):
    "Frames actually present on disk (full on Kaggle; possibly partial locally)."
    T = zarr.open(str(zp), mode="r")["0"].shape[0]                  # (T, Z, Y, X)
    present = [t for t in range(T) if (Path(zp) / "0" / "c" / str(t) / "0" / "0" / "0").exists()]
    return max(present) + 1 if present else 0

def run_submission(cfg, data_dir=TEST_PATH):
    "Track every movie under data_dir -> submission-ready DataFrame (index name 'id')."
    cache = {}
    def _load(ds):                                                  # ds -> (T,Z,Y,X) array, cached
        key = str(Path(data_dir) / (ds + ".zarr"))
        if key not in cache: cache[key] = zarr.open(key, mode="r")["0"]
        return cache[key]

    parts, stats = [], []
    for zp in sorted(Path(data_dir).glob("*.zarr")):
        ds = zp.name.replace(".zarr", "")
        if not (zp / "0" / "zarr.json").exists():
            print("  (skip, no metadata)", ds); continue
        T = avail_T(zp)
        if T == 0:
            print("  (skip, no frames present)", ds); continue
        t0 = time.time()
        nodes, edges, st = track_video(dataset=ds, load_volume=_load, T=T, cfg=cfg)
        st["sec"] = round(time.time() - t0, 1); stats.append(st); parts += [nodes, edges]
        print(f"  {ds}: T={T} nodes={st['n_nodes']} edges={st['n_edges']} "
              f"cells/frame={st['cells_per_frame']:.1f} div={st['n_div']} ({st['sec']}s)")

    sub = pd.concat(parts, ignore_index=True); sub.index.name = "id"
    return sub, pd.DataFrame(stats)

submission, run_stats = run_submission(cfg)
submission.to_csv(OUT)
print("\nwrote", OUT, "rows:", len(submission))
run_stats

In [ ]:
# Schema sanity-check against the expected columns (mirrors the reference notebook)
exp = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
assert list(submission.columns) == exp, submission.columns
_nodes = submission[submission.row_type == "node"]; _edges = submission[submission.row_type == "edge"]
assert (_nodes[["source_id", "target_id"]] == -1).all().all()
assert (_edges[["node_id", "t", "z", "y", "x"]] == -1).all().all()
# every edge endpoint must be a real node id within its dataset
ok = True
for ds, g in submission.groupby("dataset"):
    ids = set(g[g.row_type == "node"].node_id); e = g[g.row_type == "edge"]
    if not (set(e.source_id) | set(e.target_id)).issubset(ids):
        ok = False; print("DANGLING EDGE in", ds)
print("schema OK:", ok, "| node rows:", len(_nodes), "| edge rows:", len(_edges),
      "| datasets:", submission.dataset.nunique())
submission.head(6)